# 05 — Rolling Backtesting and Report-Specific Seasonality

This notebook explains and demonstrates the dynamic-seasonality forecasting framework.

## What this notebook covers

| Topic | Section |
|---|---|
| 28-day forecast horizon | §1 objective |
| Why horizon ≠ seasonal period | §4 |
| Per-fold seasonality diagnostics | §5–6 |
| Candidate model registry | §7 |
| Fold-level forecast results | §8 |
| Evaluation metrics | §9 |
| Horizon-bucket breakdown | §10 |
| Cross-fold model–period summaries | §11 |
| Joint selection of model and m | §12 |
| Full-history production refit | §13 |
| Portfolio seasonality outputs | §14 |
| Limitations | §15 |

**Important:** This notebook imports all reusable logic from `src/`.  
It does **not** reimplement seasonality, backtesting, metric, selection, or production-refit logic.

## 0. Imports and Configuration

In [ ]:
import uuid
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Pipeline utilities ───────────────────────────────────────────────────────
from src.pipelines.run_forecasting_pipeline import (
    STANDARD_DATE_COL,
    STANDARD_REPORT_ID_COL,
    STANDARD_TARGET_COL,
    MIN_DAYS,
    MIN_NONZERO_DAYS,
    MIN_NONZERO_RATIO,
    MIN_TOTAL_VIEWS,
    build_daily_series_for_all_reports,
    filter_by_data_criteria,
    get_project_root,
    load_canonical_daily_series,
    load_forecast_feature_input,
    run_data_quality_checks,
    save_production_outputs,
    validate_forecasting_series_input,
)

# ── Forecasting configuration ─────────────────────────────────────────────────
from src.config.forecasting import (
    BACKTEST_FOLDS,
    BACKTEST_STEP_DAYS,
    FORECAST_HORIZON_DAYS,
    MAX_SEASONAL_CANDIDATES_PER_FOLD,
    MIN_SEASONAL_CYCLES,
    MIN_TRAIN_DAYS,
    MIN_VALID_FOLDS,
    NON_SEASONAL_PERIOD,
    SEASONAL_CANDIDATES,
)

# ── Seasonality profiling ─────────────────────────────────────────────────────
from src.models.seasonality import SeasonalityProfile, profile_seasonality

# ── Rolling-origin splits ─────────────────────────────────────────────────────
from src.models.backtesting import ForecastFold, generate_rolling_splits

# ── Candidate models ──────────────────────────────────────────────────────────
from src.models.candidates import (
    ModelResult,
    forecast_auto_arima,
    forecast_ets,
    forecast_moving_average,
    forecast_naive,
    forecast_seasonal_naive,
)

# ── Fold evaluation (candidate-aware) ────────────────────────────────────────
from src.models.backtest_evaluation import (
    BacktestConfig,
    evaluate_candidates_across_folds,
)

# ── Horizon-bucket metrics ────────────────────────────────────────────────────
from src.models.horizon_evaluation import HORIZON_BUCKETS, calculate_horizon_bucket_metrics

# ── Cross-fold candidate summary ──────────────────────────────────────────────
from src.models.model_summary import summarise_candidate_performance

# ── Model selection ───────────────────────────────────────────────────────────
from src.models.selection import (
    MAX_BIAS_RATIO,
    MODEL_COMPLEXITY,
    RELATIVE_IMPROVEMENT_TOLERANCE,
    select_candidate_models,
)

# ── Production forecast ───────────────────────────────────────────────────────
from src.models.production_forecast import (
    PRODUCTION_FORECAST_COLS,
    build_production_forecast,
)

# ── Seasonality diagnostics ───────────────────────────────────────────────────
from src.models.seasonality_diagnostics import (
    CANDIDATE_COLS,
    SUMMARY_COLS,
    build_seasonality_candidates,
    build_seasonality_summary,
    save_seasonality_diagnostics,
    validate_seasonality_candidates,
    validate_seasonality_summary,
)

In [ ]:
PROJECT_ROOT = get_project_root()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR    = PROJECT_ROOT / "outputs"
DIAG_DIR      = OUTPUT_DIR / "diagnostics"

RUN_TIMESTAMP = pd.Timestamp.now()
RUN_ID = RUN_TIMESTAMP.strftime("%Y%m%d_%H%M%S") + "_" + str(uuid.uuid4())[:8]

# ── Demonstration controls ────────────────────────────────────────────────────
# Pin a specific report to the per-report visualisations; None = auto-select
# the longest eligible series deterministically.
DEMO_REPORT_ID: str | None = None

# Set False to include ETS and Auto-ARIMA (requires statsmodels and pmdarima).
USE_FAST_MODELS_ONLY: bool = True

# Cap on reports evaluated in sections 11–12 (increase for a full portfolio run).
MAX_REPORTS_FOR_SUMMARY: int = 5

# DEMO_MODE_SAVE: set True to write production forecast outputs during this
# notebook run.  Leave False to avoid overwriting production forecast history.
DEMO_MODE_SAVE: bool = False

DEMO_BACKTEST_CONFIG = BacktestConfig(
    horizon=FORECAST_HORIZON_DAYS,
    n_folds=BACKTEST_FOLDS,
    step=BACKTEST_STEP_DAYS,
    min_train_size=MIN_TRAIN_DAYS,
)

print(f"RUN_ID:                    {RUN_ID}")
print(f"PROJECT_ROOT:              {PROJECT_ROOT.resolve()}")
print(f"FORECAST_HORIZON_DAYS:     {FORECAST_HORIZON_DAYS}")
print(f"BACKTEST_FOLDS:            {BACKTEST_FOLDS}")
print(f"SEASONAL_CANDIDATES:       {SEASONAL_CANDIDATES}")
print(f"MIN_SEASONAL_CYCLES:       {MIN_SEASONAL_CYCLES}")
print(f"MIN_TRAIN_DAYS:            {MIN_TRAIN_DAYS}")
print(f"MIN_VALID_FOLDS:           {MIN_VALID_FOLDS}")
print(f"USE_FAST_MODELS_ONLY:      {USE_FAST_MODELS_ONLY}")
print(f"DEMO_MODE_SAVE:            {DEMO_MODE_SAVE}")

## 1. Evaluation Objective

### 28-day forecast horizon

Each production forecast covers the next **28 days** (four full calendar weeks).

### Horizon ≠ seasonal period

The forecast horizon does **not** determine the seasonal period.  
Each report is profiled independently using its own training history:

| Report pattern | Seasonal period used (m) |
|---|---|
| Strong day-of-week cycle | m = 7 (weekly) |
| Biweekly reporting cadence | m = 14 |
| Four-week usage rhythm | m = 28 |
| Approximate monthly | m = 30 |
| Approximate quarterly | m = 90 |
| No detectable seasonality | m = 1 (non-seasonal) |

A weekly report uses m = 7 **within a 28-day forecast**. The period is a model  
parameter, not the horizon.

### How the framework selects m

1. For each rolling fold, `profile_seasonality` analyses the **fold training window only**  
   (no test leakage) and proposes candidate values of m.
2. `evaluate_candidates_across_folds` runs every (model_family, candidate_m) pair  
   on every fold.
3. `summarise_candidate_performance` aggregates per (report, family, m).
4. `select_candidate_models` applies the selection policy and records the final  
   `(selected_model_family, selected_model_name, selected_m)` triple.
5. `build_production_forecast` refits on full history with the **exact selected m**.

## 2. Load the Canonical Daily Series

The pipeline accepts **only** `mart_report_daily_series.csv`.  
No fallback input files, no inline aggregation, no synthetic data generation.

In [ ]:
daily_series_df, report_views_df, active_series_input_file = load_forecast_feature_input(PROJECT_ROOT)

print(f"Loaded:       {active_series_input_file.relative_to(PROJECT_ROOT)}")
print(f"Shape:        {daily_series_df.shape}")
print(f"Reports:      {daily_series_df[STANDARD_REPORT_ID_COL].nunique()}")
print(f"Date range:   {daily_series_df[STANDARD_DATE_COL].min().date()} → "
      f"{daily_series_df[STANDARD_DATE_COL].max().date()}")

dq = run_data_quality_checks(daily_series_df)
display(dq)

In [ ]:
series_dict, name_lookup, provenance = build_daily_series_for_all_reports(report_views_df)
passing_ids, data_diag = filter_by_data_criteria(series_dict, provenance)

if not passing_ids:
    raise RuntimeError(
        "No reports passed eligibility criteria.\n"
        "Run notebooks/04_feature_engineering.ipynb first."
    )

profile_cols = [
    "report_id", "n_obs", "date_min", "date_max",
    "zero_share", "total_views", "passes_data_criteria",
]
print(f"Total reports: {len(series_dict)}   |   Passing eligibility: {len(passing_ids)}")
display(
    data_diag[[c for c in profile_cols if c in data_diag.columns]]
    .sort_values("passes_data_criteria", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

In [ ]:
# History-length summary
lengths = {rid: len(s) for rid, s in series_dict.items() if rid in passing_ids}
lengths_s = pd.Series(lengths)
print("History length (days) — eligible reports:")
print(lengths_s.describe().round(1).to_string())

## 3. Evaluation Design

### Rolling-origin expanding windows

```
Fold 1:  [──────── train ────────] [── test 28d ──]
Fold 2:  [──────────────── train ────────────────] [── test 28d ──]
Fold 3:  [────────────────────────── train ───────────────────────] [── test 28d ──]
Fold 4:  [──────────────────────────────── train ─────────────────────────────────] [── test 28d ──]
```

Each fold's training window expands by `BACKTEST_STEP_DAYS` (28 days).  
Test windows never overlap — each covers exactly `FORECAST_HORIZON_DAYS` (28) days.

### Configuration constants

| Parameter | Value | Meaning |
|---|---|---|
| `FORECAST_HORIZON_DAYS` | 28 | Test-window width |
| `BACKTEST_FOLDS` | 4 | Rolling folds per report |
| `BACKTEST_STEP_DAYS` | 28 | Stride between cutoff dates |
| `MIN_TRAIN_DAYS` | 180 | Minimum training days for first fold |
| `MIN_VALID_FOLDS` | 3 | Minimum successful folds for model selection |
| `SEASONAL_CANDIDATES` | (7,14,28,30,90) | Candidate periods screened per fold |
| `MIN_SEASONAL_CYCLES` | 3 | Minimum complete cycles for eligibility |
| `MAX_SEASONAL_CANDIDATES_PER_FOLD` | 3 | Cap on profiler shortlist per fold |

In [ ]:
# Select demo report deterministically: longest passing series
if DEMO_REPORT_ID is not None and DEMO_REPORT_ID in passing_ids:
    demo_rid = DEMO_REPORT_ID
else:
    demo_rid = max(passing_ids, key=lambda r: len(series_dict[r]))

demo_series = series_dict[demo_rid]
demo_name   = name_lookup.get(demo_rid, demo_rid)

print(f"Demo report: {demo_name!r}  ({demo_rid})")
print(f"History:     {demo_series.index.min().date()} → {demo_series.index.max().date()} "
      f"({len(demo_series)} days)")

folds, split_status = generate_rolling_splits(
    demo_series,
    horizon=DEMO_BACKTEST_CONFIG.horizon,
    n_folds=DEMO_BACKTEST_CONFIG.n_folds,
    step=DEMO_BACKTEST_CONFIG.step,
    min_train_size=DEMO_BACKTEST_CONFIG.min_train_size,
)
print(f"Folds generated: {len(folds)}")

# Gantt-style fold diagram
fig, ax = plt.subplots(figsize=(13, max(3, len(folds) * 0.9 + 1.5)))
_TRAIN_CLR, _TEST_CLR = "steelblue", "tomato"
for fold in folds:
    y = fold.fold_number
    ax.barh(y, (fold.train_end - fold.train_start).days,
            left=fold.train_start, color=_TRAIN_CLR, alpha=0.65, height=0.55)
    ax.barh(y, (fold.test_end - fold.test_start).days,
            left=fold.test_start, color=_TEST_CLR, alpha=0.9, height=0.55)
    ax.text(fold.cutoff_date, y, f" cutoff\n {fold.cutoff_date.date()}",
            va="center", fontsize=7, color="dimgray")

handles = [
    mpatches.Patch(color=_TRAIN_CLR, alpha=0.65, label="Training window (expanding)"),
    mpatches.Patch(color=_TEST_CLR, alpha=0.9, label=f"Test window ({FORECAST_HORIZON_DAYS} days)"),
]
ax.legend(handles=handles, loc="upper left")
ax.set_yticks([f.fold_number for f in folds])
ax.set_yticklabels([f"Fold {f.fold_number}  ({len(f.train_series)}d train)" for f in folds])
ax.set_xlabel("Date")
ax.set_title(f"Rolling-origin fold windows — {demo_name!r}")
ax.xaxis_date()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Forecast Horizon vs Seasonal Period

These are **two independent concepts**.

| Concept | Value | Determined by |
|---|---|---|
| Forecast horizon | 28 days | Fixed portfolio standard |
| Seasonal period m | 1, 7, 14, 28, 30, or 90 | Per-report backtesting |

### Concrete examples

```
Weekly report (m=7):
  Model: SARIMA(p,d,q)(P,D,Q)[7]
  Training uses weekly pattern.
  Output: 28-day forecast  (= 4 × m)

Monthly-like report (m=30):
  Model: Seasonal Naive m=30
  Repeats the most recent 30-day pattern.
  Output: 28-day forecast  (< m — the model still forecasts 28 days)

Non-seasonal report (m=1):
  Model: ARIMA (no seasonal terms) or Naive
  No seasonal lags used.
  Output: 28-day forecast
```

> **m does not need to equal or divide the forecast horizon.**  
> A seasonal naive model with m=30 produces 28 forecast values by taking  
> the first 28 positions from one repetition of the 30-day pattern.

## 5. Seasonality Diagnostics

`profile_seasonality` analyses the fold training window and computes for each candidate m:

- **cycles_available** — `floor(n_train / m)`: must be ≥ `MIN_SEASONAL_CYCLES`
- **autocorrelation_at_m** — ACF at lag m (clamped to 0 before normalisation)
- **spectral_power_at_m** — periodogram power at frequency 1/m as a share of total
- **candidate_score** — `0.5 × acf_norm + 0.5 × spectral_norm` (0–1)

The profiler returns a **shortlist** (highest-scoring eligible candidates).  
Backtesting then decides which period is actually best.

In [ ]:
def _show_seasonality_profile(
    rid: str,
    series: pd.Series,
    name: str,
    fold_idx: int = 0,
) -> SeasonalityProfile:
    """Profile the first fold training window and display diagnostic signals."""
    folds_tmp, _ = generate_rolling_splits(
        series,
        horizon=DEMO_BACKTEST_CONFIG.horizon,
        n_folds=DEMO_BACKTEST_CONFIG.n_folds,
        step=DEMO_BACKTEST_CONFIG.step,
        min_train_size=DEMO_BACKTEST_CONFIG.min_train_size,
    )
    if fold_idx >= len(folds_tmp):
        fold_idx = 0
    fold = folds_tmp[fold_idx]
    profile = profile_seasonality(fold.train_series, candidate_periods=SEASONAL_CANDIDATES)

    print(f"\n{'='*60}")
    print(f"Report: {name!r}  ({rid})")
    print(f"  Training window: {fold.train_start.date()} → {fold.train_end.date()}"
          f"  ({len(fold.train_series)} days)")
    print(f"  Seasonality status : {profile.seasonality_status}")
    print(f"  Dominant period    : {profile.dominant_detected_period}")
    print(f"  Shortlisted m      : {profile.selected_candidate_periods}")

    rows = []
    for m in SEASONAL_CANDIDATES:
        rows.append({
            "m": m,
            "cycles_available": profile.cycles_available_by_period.get(m, 0),
            "acf": round(profile.autocorrelation_by_period.get(m, float("nan")), 4),
            "spectral_power": round(profile.spectral_power_by_period.get(m, float("nan")), 4),
            "score": round(profile.candidate_score_by_period.get(m, float("nan")), 4),
            "eligible": m not in profile.excluded_periods,
            "shortlisted": m in profile.selected_candidate_periods,
            "exclusion_reason": profile.exclusion_reason_by_period.get(m, ""),
        })
    diag_df = pd.DataFrame(rows)
    display(diag_df)

    # Daily usage plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
    axes[0].plot(series.index, series.values, color="steelblue", linewidth=0.8, alpha=0.85)
    axes[0].set_title(f"Daily usage — {name!r}")
    axes[0].set_xlabel("Date")
    axes[0].set_ylabel("Daily views")

    # Candidate scores bar chart
    eligible_rows = diag_df[diag_df["eligible"]]
    if not eligible_rows.empty:
        bars = axes[1].bar(
            eligible_rows["m"].astype(str),
            eligible_rows["score"],
            color=["tomato" if sl else "steelblue"
                   for sl in eligible_rows["shortlisted"]],
            alpha=0.8,
        )
        axes[1].set_xlabel("Candidate m")
        axes[1].set_ylabel("Composite score")
        axes[1].set_title("Candidate scores (red = shortlisted)")
        axes[1].set_ylim(0, 1.05)
    else:
        axes[1].text(0.5, 0.5, "No eligible candidates",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
    return profile

In [ ]:
# ── Representative reports ────────────────────────────────────────────────────
# We need at least: weekly, monthly-like, non-seasonal, quarterly-excluded.
# Select deterministically: sort by series length desc, pick first MAX_REPORTS.

eligible_sorted = sorted(passing_ids, key=lambda r: len(series_dict[r]), reverse=True)
demo_reports = eligible_sorted[:4]  # up to 4 representative reports

print("Representative reports for diagnostics:")
for i, rid in enumerate(demo_reports):
    n = len(series_dict[rid])
    print(f"  [{i}] {rid}  ({n} days)  name={name_lookup.get(rid, rid)!r}")

In [ ]:
# Show diagnostics for each representative report
_profiles = {}
for rid in demo_reports:
    _profiles[rid] = _show_seasonality_profile(
        rid, series_dict[rid], name_lookup.get(rid, rid)
    )

In [ ]:
# ── Summary: which periods appeared ──────────────────────────────────────────
status_rows = []
for rid in demo_reports:
    p = _profiles[rid]
    status_rows.append({
        "report_id": rid[:12] + "…" if len(rid) > 12 else rid,
        "history_days": len(series_dict[rid]),
        "status": p.seasonality_status,
        "dominant_m": p.dominant_detected_period,
        "shortlisted_m": [m for m in p.selected_candidate_periods if m > NON_SEASONAL_PERIOD],
        "excluded_m": list(p.excluded_periods.keys()),
    })
display(pd.DataFrame(status_rows))

## 6. Leakage Prevention

The design guarantees that test-window observations **cannot** influence which  
seasonal periods are evaluated:

1. `generate_rolling_splits` produces `(train_series, test_series)` pairs.
2. `_build_fold_candidates(fold, ...)` calls `profile_seasonality(fold.train_series, ...)`.  
   Only `fold.train_series` is passed — `fold.test_series` is never read during profiling.
3. The profiler returns a shortlist of candidate m values.
4. Candidate models are fitted **only on `train_series`** and evaluated on `test_series`.

```python
# Simplified internal logic (do not copy — use evaluate_candidates_across_folds)
for fold in folds:
    profile = profile_seasonality(fold.train_series)     # ← train only
    candidates = build_candidates_from_profile(profile)
    for spec in candidates:
        result = spec.model_fn(fold.train_series, horizon)   # ← train only
        errors = result.forecast - fold.test_series          # ← now we read test
```

In [ ]:
# Demonstrate per-fold candidate variation on the demo report
folds_demo, _ = generate_rolling_splits(
    demo_series,
    horizon=DEMO_BACKTEST_CONFIG.horizon,
    n_folds=DEMO_BACKTEST_CONFIG.n_folds,
    step=DEMO_BACKTEST_CONFIG.step,
    min_train_size=DEMO_BACKTEST_CONFIG.min_train_size,
)

print(f"Fold-specific candidate shortlists for {demo_name!r}:\n")
fold_cand_rows = []
for fold in folds_demo:
    profile = profile_seasonality(fold.train_series, candidate_periods=SEASONAL_CANDIDATES)
    shortlisted = [m for m in profile.selected_candidate_periods if m > NON_SEASONAL_PERIOD]
    fold_cand_rows.append({
        "fold": fold.fold_number,
        "train_days": len(fold.train_series),
        "cutoff": fold.cutoff_date.date(),
        "status": profile.seasonality_status,
        "shortlisted_m": shortlisted or ["(none — non-seasonal)"],
        "dominant_m": profile.dominant_detected_period,
    })

display(pd.DataFrame(fold_cand_rows))
print("\nNote: shortlisted_m can differ across folds as training history grows.")

## 7. Candidate Models

`evaluate_candidates_across_folds` builds a per-fold candidate list combining:

- **Baseline candidates** (always evaluated, m=1): naive, moving_average, auto_arima_m1, ets_m1
- **Seasonal candidates** (from profiler shortlist): seasonal_naive_m{m}, auto_arima_m{m}, ets_m{m}

Each model function shares the interface `(training_series, horizon, seasonal_period=m) → ModelResult`.

In [ ]:
candidate_registry_info = pd.DataFrame([
    {"model_family": "naive",          "candidate_source": "baseline",
     "example_name": "naive",
     "complexity": MODEL_COMPLEXITY["naive"],
     "requires_extra_pkg": False,
     "description": "Last-value constant forecast."},
    {"model_family": "moving_average", "candidate_source": "baseline",
     "example_name": "moving_average",
     "complexity": MODEL_COMPLEXITY["moving_average"],
     "requires_extra_pkg": False,
     "description": "Mean of most recent seasonal window."},
    {"model_family": "auto_arima",    "candidate_source": "baseline (m=1)",
     "example_name": "auto_arima_m1",
     "complexity": MODEL_COMPLEXITY["auto_arima"],
     "requires_extra_pkg": True,
     "description": "Non-seasonal ARIMA (pmdarima)."},
    {"model_family": "ets",           "candidate_source": "baseline (m=1)",
     "example_name": "ets_m1",
     "complexity": MODEL_COMPLEXITY["ets"],
     "requires_extra_pkg": True,
     "description": "Non-seasonal ETS (statsmodels)."},
    {"model_family": "seasonal_naive", "candidate_source": "profiler",
     "example_name": "seasonal_naive_m7 / …_m30",
     "complexity": MODEL_COMPLEXITY["seasonal_naive"],
     "requires_extra_pkg": False,
     "description": "Repeats last observed m-day pattern."},
    {"model_family": "auto_arima",    "candidate_source": "profiler",
     "example_name": "auto_arima_m7 / auto_arima_m30",
     "complexity": MODEL_COMPLEXITY["auto_arima"],
     "requires_extra_pkg": True,
     "description": "SARIMA with profiler-selected m (pmdarima)."},
    {"model_family": "ets",           "candidate_source": "profiler",
     "example_name": "ets_m7 / ets_m30",
     "complexity": MODEL_COMPLEXITY["ets"],
     "requires_extra_pkg": True,
     "description": "Seasonal ETS with profiler-selected m (statsmodels)."},
])
display(candidate_registry_info)
print(f"\nActive in this run: USE_FAST_MODELS_ONLY={USE_FAST_MODELS_ONLY}")
if USE_FAST_MODELS_ONLY:
    print("  → auto_arima and ets families excluded (set USE_FAST_MODELS_ONLY=False to include).")

## 8. Fold-Level Forecast Results

`evaluate_candidates_across_folds` runs every (model_family, candidate_m) pair  
on every fold and returns a `predictions` table and a `fold_metrics` table.

In [ ]:
print(f"Running candidate evaluation for {demo_name!r}…")
demo_predictions, demo_fold_metrics = evaluate_candidates_across_folds(
    report_id=demo_rid,
    series=demo_series,
    backtest_config=DEMO_BACKTEST_CONFIG,
    candidate_periods=SEASONAL_CANDIDATES,
    include_ets=not USE_FAST_MODELS_ONLY,
    include_arima=not USE_FAST_MODELS_ONLY,
)
print(f"  Prediction rows:  {len(demo_predictions):,}")
print(f"  Fold-metric rows: {len(demo_fold_metrics):,}")
print(f"  Candidates seen:  {demo_fold_metrics['model_name'].unique().tolist()}")

In [ ]:
# Actual-vs-forecast chart for the last fold
last_fold = folds_demo[-1]
fold_preds = demo_predictions[
    demo_predictions["fold_number"] == last_fold.fold_number
].copy()
fold_preds["forecast_date"] = pd.to_datetime(fold_preds["forecast_date"])

# Limit to one candidate per family+m to keep chart readable
shown_models = (
    fold_preds[["model_name", "candidate_m"]]
    .drop_duplicates()
    .sort_values(["candidate_m", "model_name"])
    .head(5)["model_name"]
    .tolist()
)

context_start = last_fold.train_end - pd.Timedelta(days=28)
context = demo_series.loc[context_start:last_fold.train_end]

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(context.index, context.values,
        color="dimgray", linewidth=1.5, label="Observed (training context)")

# Actuals
actuals_df = fold_preds[fold_preds["model_name"] == fold_preds["model_name"].iloc[0]]
ax.plot(actuals_df["forecast_date"], actuals_df["actual"],
        color="black", linewidth=2.0, linestyle="--", label="Actual (test window)")

_COLOURS = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, model_name in enumerate(shown_models):
    mdf = fold_preds[fold_preds["model_name"] == model_name]
    if mdf.empty or mdf["forecast"].isna().all():
        continue
    m_val = int(mdf["candidate_m"].iloc[0])
    ax.plot(mdf["forecast_date"], mdf["forecast"],
            label=f"{model_name} (m={m_val})",
            color=_COLOURS[i % len(_COLOURS)], linewidth=1.4, alpha=0.85)

ax.axvline(last_fold.cutoff_date, color="gray", linestyle=":",
           linewidth=1.0, label=f"Cutoff (fold {last_fold.fold_number})")
ax.set_title(f"Fold {last_fold.fold_number} — actual vs candidate forecasts\n{demo_name!r}")
ax.set_xlabel("Date")
ax.set_ylabel("Daily views")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

## 9. Evaluation Metrics

| Metric | Formula | Interpretation |
|---|---|---|
| MAE | mean |actual − forecast| | Scale-dependent absolute error |
| WAPE | Σ|error| / Σ|actual| | Weighted percentage error (robust to near-zero actuals) |
| MASE (lag-1) | MAE / mean |y_t − y_{t-1}| | Scale-free vs. random-walk baseline |
| bias | mean (forecast − actual) | Systematic over/under-forecast |
| interval_coverage | P(actual ∈ [lower, upper]) | Empirical 95 % PI coverage |
| mean_interval_width | mean (upper − lower) | PI width (narrower = more precise) |

### Why the same MASE denominator for all candidates?

`mase_lag1` uses `mean |y_t − y_{t-1}|` computed from the **fold's training series only**.  
This denominator is **identical for every (model, m) pair in the same fold**, so MASE values  
are directly comparable across candidates — a SARIMA m=7 MASE of 0.8 means the same thing  
as a seasonal-naive m=7 MASE of 0.8.

`mase_m` (also in the table) uses each candidate's own period as the denominator.  
It is a per-candidate diagnostic, not used for cross-candidate ranking.

In [ ]:
metric_display_cols = [
    "fold_number", "model_name", "candidate_m",
    "mase_lag1", "wape", "mae", "bias",
    "interval_coverage", "fit_status",
]
demo_metric_display = (
    demo_fold_metrics[[c for c in metric_display_cols if c in demo_fold_metrics.columns]]
    .sort_values(["fold_number", "mase_lag1"])
    .reset_index(drop=True)
)
print(f"Fold-level metrics — {demo_name!r}  ({len(demo_metric_display)} rows):")
display(demo_metric_display)

## 10. Horizon-Bucket Performance

Forecast accuracy is decomposed across four sub-windows to reveal degradation  
with increasing lead time:

| Bucket | Steps | Typical pattern |
|---|---|---|
| days_1_7 | 1–7 | Best accuracy; recent history dominates |
| days_8_14 | 8–14 | Moderate degradation |
| days_15_28 | 15–28 | Highest uncertainty |
| full_horizon | 1–28 | Overall summary |

In [ ]:
# Use only the demo report's predictions (no series_lookup for mase denominator needed
# — the bucket function uses lag-1 from training internally)
bucket_metrics = calculate_horizon_bucket_metrics(
    predictions=demo_predictions,
    series_lookup={demo_rid: demo_series},
)

bucket_order = [b[0] for b in HORIZON_BUCKETS]
bucket_summary = (
    bucket_metrics
    .groupby(["model_name", "horizon_bucket"])[["mae", "mase_lag1"]]
    .mean()
    .reset_index()
)

models_bp = sorted(bucket_summary["model_name"].unique())
x = np.arange(len(bucket_order))
width = 0.8 / max(len(models_bp), 1)
_COLOURS = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for col_idx, (metric_key, ax) in enumerate(zip(["mae", "mase_lag1"], axes)):
    for m_idx, model_name in enumerate(models_bp):
        mdata = bucket_summary[bucket_summary["model_name"] == model_name]
        vals = [
            float(mdata.loc[mdata["horizon_bucket"] == b, metric_key].values[0])
            if len(mdata[mdata["horizon_bucket"] == b]) else float("nan")
            for b in bucket_order
        ]
        ax.bar(x + m_idx * width, vals, width, label=model_name,
               color=_COLOURS[m_idx % len(_COLOURS)], alpha=0.8)
    ax.set_xticks(x + width * (len(models_bp) - 1) / 2)
    ax.set_xticklabels(bucket_order, rotation=20, ha="right")
    ax.set_ylabel(metric_key)
    ax.set_title(f"{metric_key} by horizon bucket (mean across folds)")
    ax.legend(fontsize=7)
    if "mase" in metric_key:
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8)

plt.suptitle(f"Horizon-bucket performance — {demo_name!r}", y=1.01)
plt.tight_layout()
plt.show()

## 11. Cross-Fold Model–Period Summaries

`summarise_candidate_performance` aggregates fold-level metrics into one row per  
`(report_id, model_family, candidate_m)`.

**`auto_arima_m7` and `auto_arima_m30` are treated as separate candidates** — they  
share the same family but different periods and will have different MASE scores.

In [ ]:
# Evaluate additional reports for a richer portfolio summary
all_fold_metrics_list = [demo_fold_metrics]
additional_ids = [r for r in passing_ids if r != demo_rid][:MAX_REPORTS_FOR_SUMMARY - 1]

if additional_ids:
    print(f"Evaluating {len(additional_ids)} additional report(s)…")
    for rid in additional_ids:
        try:
            _, fm = evaluate_candidates_across_folds(
                rid, series_dict[rid], DEMO_BACKTEST_CONFIG,
                candidate_periods=SEASONAL_CANDIDATES,
                include_ets=not USE_FAST_MODELS_ONLY,
                include_arima=not USE_FAST_MODELS_ONLY,
            )
            all_fold_metrics_list.append(fm)
            print(f"  ✓ {rid}")
        except Exception as exc:
            print(f"  ✗ {rid}: {exc}")

combined_fold_metrics = pd.concat(all_fold_metrics_list, ignore_index=True)
candidate_summary = summarise_candidate_performance(combined_fold_metrics)

print(f"\nCandidate summary: {len(candidate_summary)} row(s) "
      f"across {candidate_summary['report_id'].nunique()} report(s)")

summary_display_cols = [
    "report_id", "model_family", "model_name", "candidate_m",
    "valid_folds", "failed_folds", "has_sufficient_folds",
    "median_mase", "mean_wape", "mean_bias",
    "fold_win_rate", "mean_interval_coverage",
]
display(
    candidate_summary[[c for c in summary_display_cols if c in candidate_summary.columns]]
    .sort_values(["report_id", "median_mase"])
    .head(30)
    .reset_index(drop=True)
)

## 12. Joint Selection of Model and m

`select_candidate_models` applies a four-gate policy per report:

1. **Fold sufficiency** — require ≥ `MIN_VALID_FOLDS` valid folds.
2. **Bias guardrail** — exclude candidates where `|mean_bias| / mean_mae > MAX_BIAS_RATIO`.
3. **Rank** by `median_mase` ascending.
4. **Practical tie** — within `RELATIVE_IMPROVEMENT_TOLERANCE` of the best, prefer  
   the simpler model using `MODEL_COMPLEXITY` ordering, then the shorter period.

Output columns: `selected_model_family`, `selected_model_name`, `selected_m`,  
`selection_reason`, `improvement_vs_seasonal_naive_pct`.

### Selection examples to look for

| Pattern | What to observe |
|---|---|
| SARIMA wins | `selected_model_family = auto_arima`, `selected_m > 1` |
| Seasonal naive preferred | `selected_model_family = seasonal_naive` (simpler than ARIMA) |
| ARIMA m=1 wins | `selected_m = 1`, reason mentions seasonal candidates did not improve |
| Quarterly rejected | m=90 not selected; reason mentions insufficient folds |

In [ ]:
selection = select_candidate_models(candidate_summary)

n_selected = (selection["selection_status"] == "selected").sum()
n_no_model = (selection["selection_status"] == "no_reliable_model").sum()

print(f"Reports evaluated:       {len(selection)}")
print(f"  → model selected:      {n_selected}")
print(f"  → no_reliable_model:   {n_no_model}")
print(f"\nSelection policy:")
print(f"  RELATIVE_IMPROVEMENT_TOLERANCE : {RELATIVE_IMPROVEMENT_TOLERANCE:.0%}")
print(f"  MAX_BIAS_RATIO                 : {MAX_BIAS_RATIO}")
print(f"  MIN_VALID_FOLDS                : {MIN_VALID_FOLDS}")

sel_display_cols = [
    "report_id",
    "selected_model_family", "selected_model_name", "selected_m",
    "selection_status", "valid_folds", "median_mase",
    "improvement_vs_seasonal_naive_pct", "fold_win_rate",
]
display(
    selection[[c for c in sel_display_cols if c in selection.columns]]
    .reset_index(drop=True)
)

In [ ]:
print("Selection reasons\n" + "─" * 60)
for _, row in selection.iterrows():
    icon = "✓" if row["selection_status"] == "selected" else "✗"
    fam  = row.get("selected_model_family", "—")
    m    = row.get("selected_m", "—")
    print(f"{icon} {row['report_id']}  [{fam}, m={m}]")
    print(f"   {row['selection_reason']}")

## 13. Full-History Production Refitting

After selection the **evaluation phase is complete**.

`build_production_forecast` refits a **fresh** model on the full available history  
using the exact `selected_m` from the selection output.

Key guarantees:
- `selected_m` is passed explicitly — it is never replaced by a re-detected period.
- `profile_seasonality` is **not called** during production refitting.
- Training cutoff = `series.index.max()` (all observations used).
- Forecast starts the day after the final observed date.
- Output: exactly 28 rows per report (one per horizon step).

In [ ]:
eligible_series = {rid: series_dict[rid] for rid in passing_ids}

production_df = build_production_forecast(
    selection=selection,
    series_dict=eligible_series,
    run_id=RUN_ID,
    generated_at=RUN_TIMESTAMP,
    horizon=FORECAST_HORIZON_DAYS,
    selection_run_id=RUN_ID,
    candidate_summary=candidate_summary,
)

fc_rows = production_df[production_df["horizon_step"].notna()]
print(f"Production forecast rows:   {len(fc_rows):,}")
print(f"Reports with forecasts:     {fc_rows['report_id'].nunique()}")
steps_per_report = fc_rows.groupby("report_id")["horizon_step"].count().unique().tolist()
print(f"Horizon steps per report:   {steps_per_report}")
print(f"\nOutput schema ({len(PRODUCTION_FORECAST_COLS)} columns):")
print("  " + ", ".join(PRODUCTION_FORECAST_COLS))

In [ ]:
# Production forecast chart for the demo report
demo_prod = (
    production_df[
        (production_df["report_id"] == demo_rid)
        & production_df["horizon_step"].notna()
    ].copy()
)
demo_prod["forecast_date"] = pd.to_datetime(demo_prod["forecast_date"])

if demo_prod.empty:
    print(f"No production forecast available for {demo_rid!r}.")
    print("Check selection_status above — report may have no reliable model.")
else:
    # 56-day context window
    context_start = demo_series.index.max() - pd.Timedelta(days=56)
    context = demo_series.loc[context_start:]

    sel_fam  = demo_prod["selected_model_family"].iloc[0]
    sel_name = demo_prod["selected_model_name"].iloc[0]
    sel_m    = demo_prod["selected_m"].iloc[0]
    cutoff   = str(demo_prod["training_cutoff"].iloc[0])[:10]
    start    = str(demo_prod["training_start"].iloc[0])[:10]
    fb_used  = bool(demo_prod["fallback_used"].iloc[0])

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(context.index, context.values,
            color="steelblue", linewidth=1.5, label="Observed (context)")
    ax.plot(demo_prod["forecast_date"], demo_prod["forecast"],
            color="tomato", linewidth=2.0, linestyle="--",
            label=f"Production forecast — {sel_name} (m={sel_m})")

    has_bounds = (
        "lower_bound" in demo_prod.columns
        and demo_prod["lower_bound"].notna().any()
    )
    if has_bounds:
        ax.fill_between(
            demo_prod["forecast_date"],
            demo_prod["lower_bound"].clip(lower=0),
            demo_prod["upper_bound"].clip(lower=0),
            color="tomato", alpha=0.15, label="Prediction interval",
        )

    ax.axvline(demo_series.index.max(), color="gray", linestyle=":",
               linewidth=1.0, label="Training cutoff")
    title = (
        f"Production forecast — {demo_name!r}\n"
        f"Model: {sel_name} (m={sel_m}, family={sel_fam})  |  "
        f"Training: {start} → {cutoff}"
    )
    if fb_used:
        title += "  [FALLBACK USED]"
    ax.set_title(title)
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily views")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

    print(f"\nSelection reason: {demo_prod['selection_reason'].iloc[0]}")
    print(f"Production fit status: {demo_prod['production_fit_status'].iloc[0]}")
    if fb_used:
        print(f"Fallback reason: {demo_prod['fallback_reason'].iloc[0]}")

## 14. Portfolio Seasonality Outputs

`build_seasonality_summary` and `build_seasonality_candidates` produce two governance-ready  
diagnostic DataFrames.  They are serialised to `outputs/diagnostics/` when  
`save_seasonality_diagnostics` is called.

**Four period concepts are preserved as separate columns:**

| Column | Meaning |
|---|---|
| `dominant_diagnostic_period` | m with highest composite score across folds |
| `eligible_seasonal_periods` (pipe-encoded) | Shortlisted by the profiler |
| `selected_m` | Chosen by backtesting (may differ from diagnostic dominant) |
| Production period used | Always equals `selected_m` (no re-detection during refit) |

In [ ]:
# Build both diagnostic outputs
diag_summary = build_seasonality_summary(
    series_dict=eligible_series,
    selection=selection,
    fold_metrics=combined_fold_metrics,
    production_df=production_df,
    candidate_summary=candidate_summary,
)
diag_candidates = build_seasonality_candidates(combined_fold_metrics)

# Validate schemas
validate_seasonality_summary(diag_summary)
validate_seasonality_candidates(diag_candidates)

print(f"report_seasonality_summary:    {len(diag_summary)} row(s)  columns={len(diag_summary.columns)}")
print(f"report_seasonality_candidates: {len(diag_candidates)} row(s) columns={len(diag_candidates.columns)}")
print(f"\nSummary columns:\n  {list(diag_summary.columns)}")

In [ ]:
# ── Portfolio views ───────────────────────────────────────────────────────────
print("Selected m distribution:")
display(
    diag_summary["selected_m"]
    .value_counts(dropna=False)
    .rename_axis("selected_m")
    .reset_index(name="count")
    .sort_values("selected_m")
)

print("\nSelected model family distribution:")
display(
    diag_summary["selected_model_family"]
    .value_counts(dropna=False)
    .rename_axis("family")
    .reset_index(name="count")
)

print("\nReports with no reliable model:")
no_model = diag_summary[diag_summary["selected_m"].isna()]
display(no_model[["report_id", "seasonality_status", "valid_backtest_folds", "selection_reason"]]
        .head(10).reset_index(drop=True))

print("\nReports using production fallback:")
fb_rows = diag_summary[diag_summary["fallback_used"] == True]
display(fb_rows[["report_id", "selected_model_name", "fallback_reason"]]
        .head(10).reset_index(drop=True))

In [ ]:
# ── Dominant diagnostic period vs selected_m ─────────────────────────────────
agree = (
    diag_summary["dominant_diagnostic_period"] == diag_summary["selected_m"]
).sum()
disagree = (
    diag_summary["dominant_diagnostic_period"].notna()
    & diag_summary["selected_m"].notna()
    & (diag_summary["dominant_diagnostic_period"] != diag_summary["selected_m"])
).sum()
no_dominant = diag_summary["dominant_diagnostic_period"].isna().sum()

print(f"Dominant period agrees with selected_m:  {agree}")
print(f"Dominant period differs from selected_m: {disagree}")
print(f"No dominant period (non-seasonal):       {no_dominant}")
print()
print("Summary:")
display(
    diag_summary[[
        "report_id", "seasonality_status",
        "dominant_diagnostic_period", "selected_m", "selected_model_family",
        "production_fit_status", "fallback_used",
    ]].reset_index(drop=True)
)

In [ ]:
# ── Candidates table ─────────────────────────────────────────────────────────
print(f"report_seasonality_candidates ({len(diag_candidates)} rows):")
display(
    diag_candidates[[
        "report_id", "fold_number", "candidate_m",
        "candidate_eligible", "cycles_available",
        "autocorrelation_at_m", "candidate_score", "candidate_rank",
        "models_evaluated", "valid_model_count", "failed_model_count",
    ]].head(20).reset_index(drop=True)
)

In [ ]:
# ── Optionally load pre-existing persisted outputs ────────────────────────────
summary_path = DIAG_DIR / "report_seasonality_summary.csv"
candidates_path = DIAG_DIR / "report_seasonality_candidates.csv"

if summary_path.exists():
    persisted_summary = pd.read_csv(summary_path)
    print(f"Loaded persisted summary: {summary_path.relative_to(PROJECT_ROOT)}"
          f"  ({len(persisted_summary)} rows)")
else:
    print("No persisted summary found — run the production pipeline or set DEMO_MODE_SAVE=True.")

# ── Save diagnostics (only in demo mode to avoid overwriting production outputs)
if DEMO_MODE_SAVE:
    diag_paths = save_seasonality_diagnostics(diag_summary, diag_candidates, PROJECT_ROOT)
    print(f"Saved: {diag_paths['summary'].relative_to(PROJECT_ROOT)}")
    print(f"Saved: {diag_paths['candidates'].relative_to(PROJECT_ROOT)}")
else:
    print("DEMO_MODE_SAVE=False → diagnostic files not written (set True to persist).")

## 15. Limitations

### Fixed-period approximations
- **m = 30** is a fixed 30-day period, not a calendar month.  
  True month-end effects (which vary between 28 and 31 days) require calendar  
  regressors and SARIMAX.
- **m = 90** is a fixed 90-day period, not a fiscal or calendar quarter-end.  
  Same caveat applies.

### Single primary seasonal period per model
- Each fitted model captures one dominant period.  
  Reports with multiple co-existing seasonal patterns (e.g. weekly + monthly)  
  require a model that accommodates multiple Fourier terms (e.g. SARIMAX with  
  Fourier features) or a seasonal decomposition pre-step.

### History requirements for long periods
- m = 90 requires ≥ 3 × 90 = 270 days of training history per fold.  
  On the first fold (minimum train size 180 days) m = 90 is always ineligible.
- m = 30 requires ≥ 90 days — eligible from the first fold for most reports.

### Synthetic-data assumptions
- The synthetic data from `generate_synthetic_data.py` may exhibit cleaner  
  patterns than real Power BI telemetry.  Real data will have irregular holidays,  
  deployment events, and irregular access patterns not captured here.

### Compute cost
- Dynamic per-fold, per-report candidate profiling and multi-model evaluation  
  is significantly more expensive than a fixed single-model pipeline.  
  On large portfolios, parallelisation or caching is recommended.

### Auto-ARIMA behaviour
- `USE_FAST_MODELS_ONLY = True` (default) skips ETS and Auto-ARIMA.  
  SARIMA and seasonal ETS results therefore appear only when  
  `USE_FAST_MODELS_ONLY = False` and the optional packages are installed.